# 03 · Partición

Reparte las imágenes limpias en train, val y test (70/15/15) **por grupos**, nunca por imagen, y exporta los datos en formato YOLO para el experimento local.

Kernel: **AiScope (.venv)**. Entrada: `data/interim/images_clean.parquet` y `boxes_clean.parquet` (notebook 02).

- **Grupo**: sesión de captura (mismo centro, microscopista y día), unida con cualquier sesión que comparta una imagen casi duplicada. No hay ID de paciente ni de lámina; la sesión es la aproximación más prudente.
- **Reparto**: entre miles de repartos aleatorios de grupos se elige el que mejor iguala en cada split las proporciones de imágenes, cajas por clase, especie y preparación (`aiscope.data.split`).
- **Salidas**: `data/processed/splits.parquet` y `data/processed/yolo/{campo_1280, mosaicos_640}`.

In [ ]:
import sys
try:
    import aiscope  # noqa: F401
except ModuleNotFoundError:
    raise RuntimeError(f"Kernel equivocado ({sys.executable}). Usa «AiScope (.venv)»: Kernel → Change kernel.") from None

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from PIL import Image, ImageDraw

from aiscope import style
from aiscope.style import STAGE_COLORS
from aiscope.paths import RAW_DIR, INTERIM_DIR, PROCESSED_DIR
from aiscope.data.classes import CLASSES, STAGE_ES
from aiscope.data.split import near_duplicate_pairs, build_groups, group_features, split_groups
from aiscope.data.yolo import export_dataset

style.apply()
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)

FRACCIONES = {"train": 0.70, "val": 0.15, "test": 0.15}
SEMILLA = 0
INTENTOS = 3000

images = pd.read_parquet(INTERIM_DIR / "images_clean.parquet")
boxes = pd.read_parquet(INTERIM_DIR / "boxes_clean.parquet")
print(f"excluidas por anotación masiva (notebook 02): {int(images['excluir'].sum())}")
images = images[~images["excluir"]].reset_index(drop=True)
boxes = boxes[boxes["image_id"].isin(images["image_id"])].reset_index(drop=True)
print(f"{len(images)} imágenes · {len(boxes)} cajas")

## 1. Grupos

In [ ]:
dups = near_duplicate_pairs(images, max_hamming=12)
carpeta = images.set_index("image_id")["folder"]
sesion = images.set_index("image_id")["sesion"]
dups["misma_carpeta"] = dups["a"].map(carpeta).to_numpy() == dups["b"].map(carpeta).to_numpy()
dups["misma_sesion"] = dups["a"].map(sesion).to_numpy() == dups["b"].map(sesion).to_numpy()
print(f"pares casi duplicados: {len(dups)} · entre carpetas: {int((~dups['misma_carpeta']).sum())} · entre sesiones: {int((~dups['misma_sesion']).sum())}")

grupos = build_groups(images, dups)
images["grupo"] = images["image_id"].map(grupos)
tam = images.groupby("grupo").size()
print(f"sesiones: {images['sesion'].nunique()} · grupos: {tam.size}")
display(tam.describe(percentiles=[.5, .9, .99]).to_frame("imágenes por grupo").T)
print(f"los 10 grupos más grandes suman el {tam.nlargest(10).sum() / tam.sum():.0%} de las imágenes")

## 2. Reparto

In [ ]:
feats = group_features(images, boxes, grupos, CLASSES)
asignacion, err = split_groups(feats, FRACCIONES, trials=INTENTOS, seed=SEMILLA)
images["split"] = images["grupo"].map(asignacion)
boxes = boxes.merge(images[["image_id", "split", "especie", "preparacion"]], on="image_id", how="left")
print(f"error medio relativo del reparto elegido: {err:.3f}")

# Comprobaciones anti-fuga
assert images.groupby("grupo")["split"].nunique().max() == 1, "un grupo aparece en varios splits"
assert images.groupby("folder")["split"].nunique().max() == 1, "una carpeta aparece en varios splits"
split_de = images.set_index("image_id")["split"]
assert (dups["a"].map(split_de).to_numpy() == dups["b"].map(split_de).to_numpy()).all(), "duplicados en splits distintos"
print("sin grupos, carpetas ni duplicados repartidos entre splits")

## 3. Cifras por split

In [ ]:
orden = list(FRACCIONES)
es_par = boxes["es_parasito"]
resumen = pd.DataFrame({
    "grupos": images.groupby("split")["grupo"].nunique(),
    "carpetas": images.groupby("split")["folder"].nunique(),
    "imágenes": images.groupby("split").size(),
    "cajas": boxes.groupby("split").size(),
    "parásitos": boxes[es_par].groupby("split").size(),
    "artefactos": boxes[~es_par].groupby("split").size(),
    "imágenes sin parásitos": images[images["n_parasitos"] == 0].groupby("split").size(),
    "parásitos/imagen (media)": images.groupby("split")["n_parasitos"].mean().round(2),
    "parásitos/imagen (p95)": images.groupby("split")["n_parasitos"].quantile(.95),
}).reindex(orden).fillna(0)
resumen.loc["total"] = [images["grupo"].nunique(), images["folder"].nunique(), len(images), len(boxes), int(es_par.sum()),
                        int((~es_par).sum()), int((images["n_parasitos"] == 0).sum()), round(images["n_parasitos"].mean(), 2),
                        images["n_parasitos"].quantile(.95)]
display(resumen)

In [ ]:
def tabla_split(serie_split, filas, nombre):
    t = pd.crosstab(filas, serie_split).reindex(columns=orden).fillna(0).astype(int)
    pct = (100 * t.div(t.sum(axis=1), axis=0)).round(1).add_suffix(" %")
    return pd.concat([t, pct], axis=1).rename_axis(index=nombre)

display(tabla_split(boxes["split"], boxes["stage"].map(STAGE_ES), "cajas por estadio"))
display(tabla_split(images["split"], images["especie"], "imágenes por especie"))
display(tabla_split(images["split"], images["preparacion"], "imágenes por preparación"))
display(tabla_split(images["split"], images["sample_age"].fillna("sin dato"), "imágenes por antigüedad"))

fig, ax = plt.subplots(figsize=(9, 3))
pct = pd.crosstab(boxes["stage"], boxes["split"], normalize="index").reindex(index=CLASSES, columns=orden)
left = np.zeros(len(pct))
for s, col in zip(orden, ["#7e65a8", "#65c2ca", "#282e3e"]):
    ax.barh([STAGE_ES[c] for c in pct.index], pct[s], left=left, color=col, label=s)
    left += pct[s].to_numpy()
for x in np.cumsum(list(FRACCIONES.values()))[:-1]:
    ax.axvline(x, color="k", lw=0.8, ls="--")
ax.set_xlim(0, 1); ax.set_xlabel("fracción de cajas"); ax.legend(ncol=3, loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

Combinaciones preparación × especie × estadio en test y val. Con menos de 30 cajas, el mAP de esa combinación será muy ruidoso.

In [ ]:
comb = boxes.pivot_table(index=["preparacion", "especie", "stage"], columns="split", values="x0", aggfunc="size", fill_value=0).reindex(columns=orden)
comb.index = comb.index.set_levels([STAGE_ES[s] for s in comb.index.levels[2]], level=2)
display(comb)
pocas = comb[(comb["test"] < 30) & (comb.sum(axis=1) >= 30)]
print(f"combinaciones con ≥30 cajas en total pero <30 en test: {len(pocas)}")

## 4. Guardar la partición

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
splits = images[["image_id", "folder", "grupo", "sesion", "split"]]
splits.to_parquet(PROCESSED_DIR / "splits.parquet", index=False)
print(PROCESSED_DIR / "splits.parquet", splits["split"].value_counts().reindex(orden).to_dict())

## 5. Exportación a YOLO

- **`campo_1280`**: el campo del ocular recortado en cuadrado y llevado a 1280 px, para train, val y test. Sirve para la variante A (el entrenador lo reduce a 640) y la C (1280).
- **`mosaicos_640`**: el campo llevado a 1200 px y partido en 4 mosaicos de 640 con 80 px de solape, para train y val. Es la variante B. En test se evalúa por imagen completa, uniendo los mosaicos.

Una caja cortada por el borde se conserva si queda visible al menos la mitad. Si la exportación ya existe con el mismo número de ficheros, no se repite.

In [ ]:
REEXPORTAR = False
YOLO_DIR = PROCESSED_DIR / "yolo"
ids = {s: images.loc[images["split"] == s, "image_id"].tolist() for s in orden}

def exportado(out, splits_, por_imagen):
    y = out / "data.yaml"
    if not y.exists():
        return False
    return all(len(list((out / "images" / s).glob("*.jpg"))) == len(ids[s]) * por_imagen for s in splits_)

variantes = {
    "campo_1280": dict(size=1280, tile=None, splits=orden, por_imagen=1),
    "mosaicos_640": dict(size=1200, tile=640, splits=["train", "val"], por_imagen=4),
}
for nombre, v in variantes.items():
    out = YOLO_DIR / nombre
    if REEXPORTAR or not exportado(out, v["splits"], v["por_imagen"]):
        export_dataset(RAW_DIR, images, boxes, {s: ids[s] for s in v["splits"]}, CLASSES, out, size=v["size"], tile=v["tile"])
    cuenta = {s: (len(list((out / "images" / s).glob("*.jpg"))),
                  sum(1 for f in (out / "labels" / s).glob("*.txt") if f.stat().st_size == 0)) for s in v["splits"]}
    print(nombre, {s: f"{n} imágenes, {e} sin cajas" for s, (n, e) in cuenta.items()})

Comprobación visual: una imagen de train con parásitos en las dos variantes (color por estadio).

In [ ]:
def dibuja(img_path, lbl_path):
    im = Image.open(img_path).convert("RGB")
    d = ImageDraw.Draw(im)
    W = im.width
    for line in Path(lbl_path).read_text().splitlines():
        c, x, y, w, h = line.split()
        x, y, w, h = (float(v) * W for v in (x, y, w, h))
        d.rectangle([x - w / 2, y - h / 2, x + w / 2, y + h / 2], outline=STAGE_COLORS[CLASSES[int(c)]], width=max(2, W // 320))
    return im

ejemplo = images[(images["split"] == "train") & (images["n_parasitos"] >= 4)].sample(1, random_state=1)["image_id"].iloc[0]
stem = ejemplo.replace("/", "_")
campo = dibuja(YOLO_DIR / "campo_1280/images/train" / f"{stem}.jpg", YOLO_DIR / "campo_1280/labels/train" / f"{stem}.txt")
tiles = sorted((YOLO_DIR / "mosaicos_640/images/train").glob(f"{stem}_t*.jpg"))
mos = [dibuja(t, YOLO_DIR / "mosaicos_640/labels/train" / f"{t.stem}.txt") for t in tiles]
fig, axes = plt.subplots(1, 1 + len(mos), figsize=(16, 4), gridspec_kw={"width_ratios": [2] + [1] * len(mos)})
axes[0].imshow(campo); axes[0].set_title("campo_1280", loc="left")
for a, m, t in zip(axes[1:], mos, tiles):
    a.imshow(m); a.set_title(t.stem.split("_t")[-1], fontsize=8)
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()

## 6. Detector de una clase

Segunda ronda de experimentos: sin estadio y sin artefactos. Todas las cajas de parásito son la clase `parasite` y los artefactos pasan a ser fondo. La partición es la misma.

- **`parasito_1280`**: las imágenes de `campo_1280` (enlaces duros, no ocupan más disco) con las etiquetas reasignadas.
- **`parasito_1920`**: el campo llevado a 1920 px, para comprobar si más resolución ayuda con los parásitos pequeños. El campo mide unos 2.700 px en la foto original, así que a 1280 se pierde más de la mitad del detalle.

In [ ]:
from aiscope.data.classes import SINGLE_CLASS
from aiscope.data.yolo import relabel_dataset

boxes_par = boxes[boxes["es_parasito"]].assign(class_id=0)

out = YOLO_DIR / "parasito_1920"
if REEXPORTAR or not exportado(out, orden, 1):
    export_dataset(RAW_DIR, images, boxes_par, ids, SINGLE_CLASS, out, size=1920)
relabel_dataset(YOLO_DIR / "campo_1280", YOLO_DIR / "parasito_1280", SINGLE_CLASS)

def n_cajas(out, split):
    return sum(len(f.read_text().splitlines()) for f in (out / "labels" / split).glob("*.txt"))

pd.DataFrame({
    nombre: {s: f"{len(list((YOLO_DIR / nombre / 'images' / s).glob('*.jpg')))} img · {n_cajas(YOLO_DIR / nombre, s)} cajas" for s in orden}
    for nombre in ["parasito_1280", "parasito_1920"]
} | {"parásitos anotados": boxes_par.groupby("split").size().reindex(orden).map(lambda n: f"{n} cajas")})

Comprobación visual: la misma imagen de train en las dos resoluciones, con un recorte a tamaño real para ver cuánto detalle gana 1920.

In [ ]:
def dibuja_parasitos(img_path, lbl_path):
    im = Image.open(img_path).convert("RGB")
    d = ImageDraw.Draw(im)
    W = im.width
    for line in Path(lbl_path).read_text().splitlines():
        _, x, y, w, h = line.split()
        x, y, w, h = (float(v) * W for v in (x, y, w, h))
        d.rectangle([x - w / 2, y - h / 2, x + w / 2, y + h / 2], outline=style.BRAND["purple"], width=max(2, W // 400))
    return im

vistas = {n: dibuja_parasitos(YOLO_DIR / n / "images/train" / f"{stem}.jpg", YOLO_DIR / n / "labels/train" / f"{stem}.txt")
          for n in ["parasito_1280", "parasito_1920"]}
primera = Path(YOLO_DIR / "parasito_1280/labels/train" / f"{stem}.txt").read_text().split()
cx, cy = float(primera[1]), float(primera[2])
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), gridspec_kw={"width_ratios": [2, 2, 1, 1]})
for a, (n, im) in zip(axes[:2], vistas.items()):
    a.imshow(im); a.set_title(n, loc="left")
for a, (n, im) in zip(axes[2:], vistas.items()):
    W = im.width
    lado = 96 * W // 1280
    x0, y0 = int(cx * W) - lado // 2, int(cy * W) - lado // 2
    a.imshow(im.crop((x0, y0, x0 + lado, y0 + lado))); a.set_title(f"detalle {W} px", fontsize=9)
for a in axes:
    a.axis("off")
plt.tight_layout(); plt.show()